# Week 11. 0515 와인 분류 대시보드

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050

## 1. 과제 목표

2026.05.15 과제는 와인 데이터셋을 기반으로 품종 분류 대시보드 흐름을 구성하는 것입니다.  
이 노트북에서는 데이터 조회, 클래스별 요약, 피처 중요도, 상위 피처 기반 예측 함수를 순서대로 구현합니다.

In [1]:
from math import exp
from statistics import mean

CLASS_INFO = {
    "class_0": {"name": "Barolo", "description": "알코올과 프롤린 수치가 높고 색이 진한 편입니다."},
    "class_1": {"name": "Grignolino", "description": "중간 정도의 색 강도와 균형 잡힌 페놀 특성을 보입니다."},
    "class_2": {"name": "Barbera", "description": "색 강도와 산도 관련 특성이 두드러지는 편입니다."},
}

TOP_FEATURES = [
    {"feature": "flavanoids", "importance": 0.202, "cumulative": 0.202},
    {"feature": "color_intensity", "importance": 0.171, "cumulative": 0.373},
    {"feature": "proline", "importance": 0.139, "cumulative": 0.513},
    {"feature": "alcohol", "importance": 0.112, "cumulative": 0.625},
    {"feature": "od280_od315", "importance": 0.112, "cumulative": 0.737},
    {"feature": "hue", "importance": 0.071, "cumulative": 0.807},
]

FALLBACK_ROWS = [
    {"sample_id": 1, "class_name": "class_0", "alcohol": 14.23, "flavanoids": 3.06, "color_intensity": 5.64, "proline": 1065.0, "od280_od315": 3.92, "hue": 1.04},
    {"sample_id": 2, "class_name": "class_1", "alcohol": 12.37, "flavanoids": 2.45, "color_intensity": 2.12, "proline": 520.0, "od280_od315": 2.78, "hue": 1.19},
    {"sample_id": 3, "class_name": "class_2", "alcohol": 13.17, "flavanoids": 0.63, "color_intensity": 7.90, "proline": 725.0, "od280_od315": 1.48, "hue": 0.60},
]

## 2. 기본 데이터와 상위 피처

대시보드에서 사용할 품종 설명과 상위 6개 피처를 먼저 정의합니다.  
상위 피처를 별도 리스트로 두면 화면 표시와 예측 입력을 같은 기준으로 맞출 수 있습니다.

In [2]:
def load_rows():
    try:
        from sklearn.datasets import load_wine
    except Exception:
        return FALLBACK_ROWS

    wine = load_wine()
    feature_index = {name: idx for idx, name in enumerate(wine.feature_names)}
    rows = []
    for index, values in enumerate(wine.data, start=1):
        rows.append(
            {
                "sample_id": index,
                "class_name": f"class_{int(wine.target[index - 1])}",
                "alcohol": round(float(values[feature_index["alcohol"]]), 2),
                "flavanoids": round(float(values[feature_index["flavanoids"]]), 2),
                "color_intensity": round(float(values[feature_index["color_intensity"]]), 2),
                "proline": round(float(values[feature_index["proline"]]), 2),
                "od280_od315": round(float(values[feature_index["od280/od315_of_diluted_wines"]]), 2),
                "hue": round(float(values[feature_index["hue"]]), 2),
            }
        )
    return rows

rows = load_rows()
len(rows), rows[:2]

(178,
 [{'sample_id': 1,
   'class_name': 'class_0',
   'alcohol': 14.23,
   'flavanoids': 3.06,
   'color_intensity': 5.64,
   'proline': 1065.0,
   'od280_od315': 3.92,
   'hue': 1.04},
  {'sample_id': 2,
   'class_name': 'class_0',
   'alcohol': 13.2,
   'flavanoids': 2.76,
   'color_intensity': 4.38,
   'proline': 1050.0,
   'od280_od315': 3.4,
   'hue': 1.05}])

## 3. 데이터 조회와 클래스별 요약

대시보드의 첫 화면에서는 전체 데이터를 조회하거나 특정 클래스만 필터링할 수 있어야 합니다.  
요약 함수는 클래스별 샘플 수와 주요 평균값을 계산해 화면 카드나 표에 사용할 수 있는 형태로 반환합니다.

In [3]:
def filter_rows(class_name="all"):
    rows = load_rows()
    if class_name == "all":
        return rows
    if class_name not in CLASS_INFO:
        raise ValueError("class_name은 all, class_0, class_1, class_2 중 하나여야 합니다.")
    return [row for row in rows if row["class_name"] == class_name]


def summarize_by_class():
    rows = load_rows()
    summary = []
    for class_name in CLASS_INFO:
        class_rows = [row for row in rows if row["class_name"] == class_name]
        summary.append(
            {
                "class_name": class_name,
                "name": CLASS_INFO[class_name]["name"],
                "count": len(class_rows),
                "avg_alcohol": round(mean(row["alcohol"] for row in class_rows), 2),
                "avg_color_intensity": round(mean(row["color_intensity"] for row in class_rows), 2),
                "avg_flavanoids": round(mean(row["flavanoids"] for row in class_rows), 2),
            }
        )
    return summary

summarize_by_class()

[{'class_name': 'class_0',
  'name': 'Barolo',
  'count': 59,
  'avg_alcohol': 13.74,
  'avg_color_intensity': 5.53,
  'avg_flavanoids': 2.98},
 {'class_name': 'class_1',
  'name': 'Grignolino',
  'count': 71,
  'avg_alcohol': 12.28,
  'avg_color_intensity': 3.09,
  'avg_flavanoids': 2.08},
 {'class_name': 'class_2',
  'name': 'Barbera',
  'count': 48,
  'avg_alcohol': 13.15,
  'avg_color_intensity': 7.4,
  'avg_flavanoids': 0.78}]

## 4. 예측 함수 설계

입력값은 상위 6개 피처로 제한합니다.  
실제 서비스에서는 학습 모델을 사용할 수 있지만, 이 노트북에서는 실행 안정성을 위해 점수 기반 예측 흐름을 구현했습니다.

In [4]:
def softmax(scores):
    max_score = max(scores.values())
    exps = {label: exp(score - max_score) for label, score in scores.items()}
    total = sum(exps.values())
    return {label: round(value / total, 4) for label, value in exps.items()}


def predict_wine(flavanoids, color_intensity, proline, alcohol, od280_od315, hue):
    inputs = {
        "flavanoids": flavanoids,
        "color_intensity": color_intensity,
        "proline": proline,
        "alcohol": alcohol,
        "od280_od315": od280_od315,
        "hue": hue,
    }
    invalid = [name for name, value in inputs.items() if value <= 0]
    if invalid:
        raise ValueError(f"0보다 큰 값이 필요합니다: {', '.join(invalid)}")

    scores = {
        "class_0": alcohol * 0.28 + flavanoids * 0.32 + proline / 1000 * 0.28 + od280_od315 * 0.12,
        "class_1": hue * 0.30 + flavanoids * 0.18 + od280_od315 * 0.28 + (1 / color_intensity) * 0.45,
        "class_2": color_intensity * 0.34 + (1 / flavanoids) * 0.25 + alcohol * 0.12 + (1 / hue) * 0.22,
    }
    probabilities = softmax(scores)
    prediction = max(probabilities, key=probabilities.get)
    return {
        "prediction": prediction,
        "name": CLASS_INFO[prediction]["name"],
        "description": CLASS_INFO[prediction]["description"],
        "probabilities": probabilities,
        "inputs": inputs,
    }

predict_wine(flavanoids=2.03, color_intensity=5.06, proline=747.0, alcohol=13.0, od280_od315=2.61, hue=0.96)

{'prediction': 'class_0',
 'name': 'Barolo',
 'description': '알코올과 프롤린 수치가 높고 색이 진한 편입니다.',
 'probabilities': {'class_0': 0.7446, 'class_1': 0.0264, 'class_2': 0.229},
 'inputs': {'flavanoids': 2.03,
  'color_intensity': 5.06,
  'proline': 747.0,
  'alcohol': 13.0,
  'od280_od315': 2.61,
  'hue': 0.96}}

## 5. 대시보드 payload 구성

대시보드 화면에서는 데이터 표, 요약, 피처 중요도, 예측 결과가 필요합니다.  
따라서 여러 함수를 따로 호출하지 않고 한 번에 사용할 수 있는 payload 함수를 구성합니다.

In [5]:
def dashboard_payload(flavanoids=2.03, color_intensity=5.06, proline=747.0, alcohol=13.0, od280_od315=2.61, hue=0.96):
    return {
        "rows": filter_rows("all")[:10],
        "summary": summarize_by_class(),
        "feature_importance": TOP_FEATURES,
        "prediction": predict_wine(
            flavanoids=flavanoids,
            color_intensity=color_intensity,
            proline=proline,
            alcohol=alcohol,
            od280_od315=od280_od315,
            hue=hue,
        ),
    }

payload = dashboard_payload()
payload.keys(), payload["prediction"]

(dict_keys(['rows', 'summary', 'feature_importance', 'prediction']),
 {'prediction': 'class_0',
  'name': 'Barolo',
  'description': '알코올과 프롤린 수치가 높고 색이 진한 편입니다.',
  'probabilities': {'class_0': 0.7446, 'class_1': 0.0264, 'class_2': 0.229},
  'inputs': {'flavanoids': 2.03,
   'color_intensity': 5.06,
   'proline': 747.0,
   'alcohol': 13.0,
   'od280_od315': 2.61,
   'hue': 0.96}})

## 6. 정리

0515 과제의 핵심은 와인 데이터셋을 단순히 예측하는 것에서 끝나지 않고, 데이터 조회와 피처 중요도 분석, 예측 입력을 하나의 대시보드 흐름으로 묶는 것입니다.  
이 구조를 사용하면 FastAPI나 Gradio 화면을 붙일 때도 데이터 준비 함수와 화면 표시 함수를 분리해서 관리할 수 있습니다.